# Prediksi `team_goals` dan `opp_goals`

Notebook ini mengolah:

- `train.csv` sebagai data pelatihan
- `test.csv` sebagai data yang akan diprediksi

Target yang diprediksi:

- `team_goals`
- `opp_goals`

Output akhir disimpan ke file **`tes.csv`** dengan kolom:

- `match_id`
- `team_goals`
- `opp_goals`

## Pendekatan yang dipakai

Di data ini, setiap pertandingan muncul dalam **2 baris** dengan `match_id` yang sama, yaitu satu baris untuk masing-masing tim.  
Karena itu, di notebook ini dipakai **smoothed attack-defense baseline**:

- kekuatan menyerang tiap tim dipelajari dari `train`
- kekuatan bertahan lawan juga dipelajari dari `train`
- ada prior tambahan dari kombinasi `gender` dan `tournament`
- hasil prediksi dua baris dalam satu match kemudian **direkonsiliasi** supaya lebih konsisten

Pendekatan ini ringan, stabil, dan aman dipakai sebagai baseline awal sebelum lanjut ke model yang lebih kompleks.


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Load data

In [2]:
train = pd.read_csv("../dataset/train.csv")
test = pd.read_csv("../dataset/test.csv")

print("train shape:", train.shape)
print("test shape :", test.shape)

display(train.head())
display(test.head())

train shape: (78772, 47)
test shape : (42422, 20)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1,500.0000","1,500.0000",NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0000,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,1.0000,NaN,NaN,0.0000,0.0000,NaN,0.0000,"1,500.0000","1,500.0000",NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M000002_England,M000002,1873-03-08,M,England,Scotland,1,0,Friendly,England,4,2,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,98.0000,98.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"1,500.0000","1,500.0000",NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M000002_Scotland,M000002,1873-03-08,M,Scotland,England,0,0,Friendly,England,2,4,1.0000,4.0000,-3.0000,0.0000,2.0000,-2.0000,1.0000,0.0000,98.0000,0.0000,1.0000,4.0000,0.0000,0.0000,2.0000,1.0000,0.0000,0.5000,"1,484.0000","1,516.0000",NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M000003_Scotland,M000003,1874-03-07,M,Scotland,England,1,0,Friendly,Scotland,2,1,1.0000,4.0000,-3.0000,-2.0000,2.0000,-4.0000,1.0000,-2.0000,364.0000,364.0000,1.0000,4.0000,1.0000,2.0000,2.0000,1.0000,0.0000,0.5000,"1,484.0000","1,516.0000",NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,"92,409.0000","1,283,330.0000","12,189.0952","9,197.0270","-9,999.0000",0.0000,"1,751.8957",25.7902
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,"1,283,330.0000","92,409.0000","9,197.0270","12,189.0952","-9,999.0000","1,751.8957",0.0000,25.7902
2,M034985_Comoros,M034985,2011-08-06,M,Comoros,Maldives,1,1,Indian Ocean Island Games,Seychelles,CAF,AFC,"656,024.0000","361,575.0000","1,447.9451","7,291.4660","-9,999.0000","1,517.0107","5,483.1175",25.7902
3,M034985_Maldives,M034985,2011-08-06,M,Maldives,Comoros,0,1,Indian Ocean Island Games,Seychelles,AFC,CAF,"361,575.0000","656,024.0000","7,291.4660","1,447.9451","-9,999.0000","5,483.1175","1,517.0107",25.7902
4,M034986_Réunion,M034986,2011-08-06,M,Réunion,Madagascar,1,1,Indian Ocean Island Games,Seychelles,Unknown,CAF,NaN,"21,731,053.0000",NaN,531.2654,"-9,999.0000",NaN,NaN,25.7902


## 2. Parsing tanggal dan pengecekan struktur

Kita ubah kolom `date` menjadi format datetime, lalu cek apakah benar setiap `match_id` terdiri dari 2 baris.


In [3]:
for df in (train, test):
    df["date"] = pd.to_datetime(df["date"])

print("Jumlah baris per match_id di train:")
display(train["match_id"].value_counts().describe())

print("Jumlah baris per match_id di test:")
display(test["match_id"].value_counts().describe())

Jumlah baris per match_id di train:


count   39,386.0000
mean         2.0000
std          0.0000
min          2.0000
25%          2.0000
50%          2.0000
75%          2.0000
max          2.0000
Name: count, dtype: float64

Jumlah baris per match_id di test:


count   21,211.0000
mean         2.0000
std          0.0000
min          2.0000
25%          2.0000
50%          2.0000
75%          2.0000
max          2.0000
Name: count, dtype: float64

## 3. Feature sederhana yang dipakai

Model baseline ini tidak memakai semua fitur kompleks dari train, karena sebagian fitur tersebut tidak tersedia di test.  
Jadi fokusnya adalah membangun model dari pola gol historis tim.


In [4]:
shared_cols = sorted(set(train.columns) & set(test.columns))
print("Jumlah shared columns:", len(shared_cols))
print(shared_cols)

Jumlah shared columns: 20
['Id', 'altitude_venue', 'confederation_opp', 'confederation_team', 'date', 'distance_travel_opp', 'distance_travel_team', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'gender', 'is_home', 'match_id', 'neutral', 'opponent', 'population_opp', 'population_team', 'team', 'temperature_venue', 'tournament', 'venue_country']


## 4. Fungsi baseline: smoothed attack-defense predictor

Intinya:

- `team_attack`: rata-rata gol yang dicetak sebuah tim
- `team_defense_concede`: rata-rata gol yang kebobolan sebuah tim
- `group_team_mean` dan `group_opp_mean`: prior berdasarkan kombinasi `gender` dan `tournament`
- smoothing dipakai supaya tim yang jumlah match-nya sedikit tidak terlalu ekstrem


In [5]:
def blended_predict(train_ref: pd.DataFrame, test_like: pd.DataFrame, smoothing: int = 8) -> pd.DataFrame:
    train_ref = train_ref.copy()
    test_like = test_like.copy()

    global_team = train_ref["team_goals"].mean()
    global_opp = train_ref["opp_goals"].mean()

    team_stats = (
        train_ref.groupby("team")
        .agg(
            n_matches=("team_goals", "size"),
            team_goals_sum=("team_goals", "sum"),
            opp_goals_sum=("opp_goals", "sum"),
        )
        .reset_index()
    )

    team_stats["team_attack"] = (
        team_stats["team_goals_sum"] + smoothing * global_team
    ) / (team_stats["n_matches"] + smoothing)

    team_stats["team_defense_concede"] = (
        team_stats["opp_goals_sum"] + smoothing * global_opp
    ) / (team_stats["n_matches"] + smoothing)

    group_stats = (
        train_ref.groupby(["gender", "tournament"])
        .agg(
            group_team_mean=("team_goals", "mean"),
            group_opp_mean=("opp_goals", "mean"),
        )
        .reset_index()
    )

    pred = test_like.merge(
        team_stats[["team", "team_attack", "team_defense_concede"]],
        on="team",
        how="left",
    ).rename(
        columns={
            "team_attack": "team_attack_self",
            "team_defense_concede": "team_defense_self",
        }
    )

    pred = pred.merge(
        team_stats[["team", "team_attack", "team_defense_concede"]].rename(
            columns={
                "team": "opponent",
                "team_attack": "opp_attack_from_team_table",
                "team_defense_concede": "opp_defense_from_team_table",
            }
        ),
        on="opponent",
        how="left",
    )

    pred = pred.merge(
        group_stats,
        on=["gender", "tournament"],
        how="left",
    )

    pred["team_attack_self"] = pred["team_attack_self"].fillna(global_team)
    pred["team_defense_self"] = pred["team_defense_self"].fillna(global_opp)
    pred["opp_attack_from_team_table"] = pred["opp_attack_from_team_table"].fillna(global_team)
    pred["opp_defense_from_team_table"] = pred["opp_defense_from_team_table"].fillna(global_opp)
    pred["group_team_mean"] = pred["group_team_mean"].fillna(global_team)
    pred["group_opp_mean"] = pred["group_opp_mean"].fillna(global_opp)

    is_home_num = pred["is_home"].fillna(0).astype(float)
    neutral_num = pred["neutral"].fillna(0).astype(float)

    pred["team_goals_pred"] = (
        0.42 * pred["team_attack_self"]
        + 0.33 * pred["opp_defense_from_team_table"]
        + 0.20 * pred["group_team_mean"]
        + 0.05 * global_team
        + 0.12 * is_home_num
        - 0.03 * neutral_num
    ).clip(0, 10)

    pred["opp_goals_pred"] = (
        0.42 * pred["opp_attack_from_team_table"]
        + 0.33 * pred["team_defense_self"]
        + 0.20 * pred["group_opp_mean"]
        + 0.05 * global_opp
        + 0.12 * (1 - is_home_num)
        - 0.03 * neutral_num
    ).clip(0, 10)

    out = pred[["Id", "match_id", "team_goals_pred", "opp_goals_pred"]].copy()

    # Rekonsiliasi dua baris dalam satu pertandingan
    for match_id, idx in out.groupby("match_id").groups.items():
        idx = list(idx)
        if len(idx) == 2:
            a, b = idx
            goal_a = float((out.loc[a, "team_goals_pred"] + out.loc[b, "opp_goals_pred"]) / 2.0)
            goal_b = float((out.loc[b, "team_goals_pred"] + out.loc[a, "opp_goals_pred"]) / 2.0)

            out.loc[a, "team_goals_pred"] = goal_a
            out.loc[a, "opp_goals_pred"] = goal_b
            out.loc[b, "team_goals_pred"] = goal_b
            out.loc[b, "opp_goals_pred"] = goal_a

    return out

## 5. Evaluasi sederhana dengan time-based split

Supaya evaluasinya lebih realistis, train dibagi berdasarkan waktu:

- bagian awal sebagai data latih
- bagian akhir sebagai validasi


In [6]:
match_dates = (
    train.groupby("match_id", as_index=False)["date"]
    .min()
    .sort_values("date")
    .reset_index(drop=True)
)

cut_date = match_dates.loc[int(len(match_dates) * 0.85), "date"]

train_match_ids = set(match_dates[match_dates["date"] < cut_date]["match_id"])
valid_match_ids = set(match_dates[match_dates["date"] >= cut_date]["match_id"])

train_fold = train[train["match_id"].isin(train_match_ids)].copy()
valid_fold = train[train["match_id"].isin(valid_match_ids)].copy()

print("cut_date:", cut_date.date())
print("train fold shape:", train_fold.shape)
print("valid fold shape:", valid_fold.shape)

cut_date: 2006-11-19
train fold shape: (66952, 47)
valid fold shape: (11820, 47)


In [7]:
valid_pred = blended_predict(train_fold, valid_fold, smoothing=8)

valid_eval = valid_fold[["Id", "team_goals", "opp_goals"]].merge(
    valid_pred,
    on="Id",
    how="left",
)

mae_team = mean_absolute_error(valid_eval["team_goals"], valid_eval["team_goals_pred"])
mae_opp = mean_absolute_error(valid_eval["opp_goals"], valid_eval["opp_goals_pred"])

print(f"Validation MAE team_goals: {mae_team:.6f}")
print(f"Validation MAE opp_goals : {mae_opp:.6f}")

display(valid_eval.head())

Validation MAE team_goals: 1.219484
Validation MAE opp_goals : 1.219484


,Id,team_goals,opp_goals,match_id,team_goals_pred,opp_goals_pred
0,M030485_Barbados,2,1,M030485,2.1852,1.4017
1,M030485_Bahamas,1,2,M030485,1.4017,2.1852
2,M030486_Panama,1,2,M030486,1.5248,1.5399
3,M030486_Peru,2,1,M030486,1.5399,1.5248
4,M030487_Găgăuzia,0,2,M030487,1.8336,1.6337


Dari running yang sudah saya lakukan, baseline ini menghasilkan kira-kira:

- **MAE team_goals ≈ 1.219484**
- **MAE opp_goals ≈ 1.219484**

Ini masih baseline sederhana, tetapi sudah cukup bagus untuk dijadikan titik awal.


## 6. Latih pada seluruh train lalu prediksi test

In [8]:
test_pred = blended_predict(train, test, smoothing=8)

submission = test_pred[["match_id", "team_goals_pred", "opp_goals_pred"]].rename(
    columns={
        "team_goals_pred": "team_goals",
        "opp_goals_pred": "opp_goals",
    }
)

submission.to_csv("tes.csv", index=False)

print("File tes.csv berhasil dibuat.")
display(submission.head())
print("submission shape:", submission.shape)

File tes.csv berhasil dibuat.


,match_id,team_goals,opp_goals
0,M034984,1.4706,1.6500
1,M034984,1.6500,1.4706
2,M034985,1.7357,1.5465
3,M034985,1.5465,1.7357
4,M034986,1.7216,1.6315


submission shape: (42422, 3)


## 7. Catatan akhir

Notebook ini menghasilkan file `tes.csv` dengan format:

- `match_id`
- `team_goals`
- `opp_goals`

Karena setiap pertandingan memiliki 2 baris pada test, maka `match_id` juga akan muncul 2 kali, masing-masing untuk perspektif tim yang berbeda.
